# Overall performance (Fig. 5 of the paper)

Test MAE and Pearson correlation per fold of (a) the activity prediction of the pose-aware and no-pose models and (b) the pose-reliability prediction of the pose-aware model, for the random and scaffold splits.

Data:
- activity, scaffold split: test predictions in `data/ablation_pose_range/` (same models as in the ablation notebook);
- activity, random split: per-fold values of Supplementary Table S4, computed on the HPC from the random-split test predictions (not stored locally), in `data/overall_performance/activity_random_per_fold.csv`;
- pose-reliability score, both splits: cross-docked test poses in `data/pose_reliability/` (predicted q against the target q of the true RMSD).

In [ ]:
from pathlib import Path
import sys
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kinodata" / "__init__.py").exists())
AP = REPO / "analysis_publication"
OUTPUT = AP / "output" / "overall_performance"; OUTPUT.mkdir(parents=True, exist_ok=True)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(AP))
from plot_style import apply_style, COLORS
apply_style()
K = 5
K_SIG, X0_SIG = 0.9, 4.5

In [ ]:
# activity, scaffold split: from the test predictions; random split: per-fold values of Table S4 (HPC predictions)
rows = []
for m in ["pose_aware", "no_pose"]:
    for k in range(K):
        d = pd.read_csv(AP / "data" / "ablation_pose_range" / m / f"fold_{k}" / "test_activity.csv.gz")
        rows.append({"split": "Scaffold", "model": m, "fold": k,
                     "MAE": (d.pred_activity - d.target_activity).abs().mean(),
                     "Pearson r": np.corrcoef(d.pred_activity, d.target_activity)[0, 1]})
act = pd.concat([pd.DataFrame(rows),
                 pd.read_csv(AP / "data" / "overall_performance" / "activity_random_per_fold.csv").assign(split="Random")])
# pose, both splits: predicted q against the target q of the cross-docked test poses
rows = []
for s in ["random", "scaffold"]:
    d = pd.read_csv(AP / "data" / "pose_reliability" / f"poses_{s}.csv.gz")
    d["target"] = 1 / (1 + np.exp(K_SIG * (d.true_rmsd - X0_SIG)))
    for k, g in d.groupby("fold"):
        rows.append({"split": s.capitalize(), "model": "pose_aware", "fold": k,
                     "MAE": (g.q - g.target).abs().mean(), "Pearson r": np.corrcoef(g.q, g.target)[0, 1]})
pose = pd.DataFrame(rows)
print(act.groupby(["split", "model"])[["MAE", "Pearson r"]].mean().round(3))
print(pose.groupby("split")[["MAE", "Pearson r"]].mean().round(3))

In [ ]:
BIG = {"axes.labelsize": 30, "xtick.labelsize": 27, "ytick.labelsize": 27, "legend.fontsize": 24,
       "axes.titlesize": 25, "xtick.major.size": 7, "ytick.major.size": 7}
LAB = {"pose_aware": "Pose-aware", "no_pose": "No-pose"}
SPLITS = ["Random", "Scaffold"]
rng = np.random.default_rng(0)
GAP = 1.35  # distance between the random and scaffold groups

def panel(ax, df, metric, models, ylim):
    w = 0.34 if len(models) == 2 else 0.5
    for i, s in enumerate(SPLITS):
        for j, m in enumerate(models):
            x = GAP * i + (j - (len(models) - 1) / 2) * (w + 0.06)
            v = df[(df.split == s) & (df.model == m)][metric].to_numpy()
            c = COLORS[m]
            ax.boxplot(v, positions=[x], widths=w, patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=c, alpha=0.35, edgecolor=c, lw=2),
                       medianprops=dict(color=c, lw=3), whiskerprops=dict(color=c, lw=2),
                       capprops=dict(color=c, lw=2))
            ax.scatter(x + rng.uniform(-w / 5, w / 5, len(v)), v, s=70, color=c, zorder=3,
                       edgecolor="white", lw=0.8)
    ax.set_xticks(GAP * np.arange(len(SPLITS)), SPLITS)
    ax.set_xlim(-0.65, GAP * (len(SPLITS) - 1) + 0.65)
    ax.set_ylim(*ylim)
    ax.set_ylabel(metric if metric == "MAE" else "Pearson $r$")
    ax.grid(axis="y", color="0.9")

with plt.rc_context(BIG):
    fig, axes = plt.subplots(1, 4, figsize=(24, 7), gridspec_kw={"wspace": 0.5})
    panel(axes[0], act, "MAE", ["pose_aware", "no_pose"], (0.45, 0.65))
    panel(axes[1], act, "Pearson r", ["pose_aware", "no_pose"], (0.55, 0.8))
    panel(axes[2], pose, "MAE", ["pose_aware"], (0.0, 0.4))
    panel(axes[3], pose, "Pearson r", ["pose_aware"], (0.0, 0.8))
    handles = [plt.Line2D([], [], marker="s", ls="", ms=16, color=COLORS[m], alpha=0.6, label=LAB[m]) for m in LAB]
    axes[0].legend(handles=handles, loc="upper left", frameon=False, handletextpad=0.3, borderaxespad=0.2)
    axes[2].set_yticks(np.arange(0, 0.41, 0.1))
    axes[3].set_yticks(np.arange(0, 0.81, 0.2))
    for ax, t in [(axes[0], "Activity prediction"), (axes[2], "Pose-reliability prediction")]:
        pos0, pos1 = ax.get_position(), axes[list(axes).index(ax) + 1].get_position()
        fig.text((pos0.x0 + pos1.x1) / 2, pos0.y1 + 0.05, t, ha="center", va="bottom", fontsize=30, fontweight="bold")
    for ax, lab in [(axes[0], "a"), (axes[2], "b")]:
        ax.text(-0.38, 1.10, f"({lab})", transform=ax.transAxes, fontsize=30, fontweight="bold", va="bottom")
    fig.savefig(OUTPUT / "overall_performance.png", dpi=600, bbox_inches="tight")

plt.show()